In [1]:
# Importamos las librerías necesarias para la exploración de datos y el análisis de vibraciones.
# Estas incluyen herramientas para manipulación de datos, cálculos estadísticos y transformadas de Fourier.
import os
import numpy as np
import pandas as pd

from scipy.stats import kurtosis
from scipy.fft import fft

In [2]:
# Definimos una función para extraer características relevantes de una señal de vibración. Estas características incluyen:
# - RMS (Root Mean Square): una medida de la magnitud de la señal.
# - Peak: el valor máximo absoluto de la señal.
# - Crest Factor: la relación entre el valor máximo y el RMS, que indica la "picosidad" de la señal.
# - Kurtosis: una medida de la "apuntamiento" de la distribución de la señal, que puede indicar la presencia de picos o colas.
# - Dominant Frequency: la frecuencia con la mayor amplitud en el espectro de Fourier, que puede indicar la frecuencia de vibración dominante.
# - Dominant Amplitude: la amplitud de la frecuencia dominante, que puede indicar la intensidad de la vibración en esa frecuencia.
# La función devuelve una lista con estas características, que pueden ser utilizadas para análisis posteriores o para alimentar modelos de machine learning.
# La función toma como entrada una señal de vibración (un array de valores) y realiza los cálculos necesarios para extraer las características mencionadas.
# Es importante destacar que estas características son comunes en el análisis de vibraciones y pueden ser útiles para detectar anomalías o patrones en los datos de vibración.
# La función utiliza la biblioteca NumPy para cálculos numéricos, SciPy para estadísticas y transformadas de Fourier, lo que facilita el proceso de extracción de características.
# En resumen, esta función es una herramienta clave para convertir datos de vibración crudos en un conjunto de características que pueden ser analizadas o utilizadas en modelos predictivos.
# La función se puede aplicar a cada señal de vibración en el dataset para construir un conjunto de características que representen las vibraciones medidas por los sensores.
# Es importante asegurarse de que la señal de entrada esté correctamente preprocesada (por ejemplo, centrada y normalizada) antes de aplicar esta función para obtener resultados más precisos.
# En general, esta función es un paso fundamental en el proceso de análisis de vibraciones y puede ser adaptada o ampliada según las necesidades específicas del proyecto o los datos disponibles.
# En el contexto de un proyecto de análisis de vibraciones, esta función puede ser utilizada para extraer características de las señales de vibración medidas por los sensores,
# lo que puede ayudar a identificar patrones, detectar anomalías o alimentar modelos de machine learning para tareas como clasificación o predicción.

def extract_features(signal):

    rms = np.sqrt(np.mean(signal**2))

    peak = np.max(np.abs(signal))

    if rms == 0:
        crest = 0
    else:
        crest = peak / rms

    kurt = kurtosis(signal)

    std = np.std(signal)
    
    mean_abs = np.mean(np.abs(signal))

    signal = signal - np.mean(signal)

    fft_values = np.abs(fft(signal))

    N = len(signal)

    freqs = np.fft.fftfreq(N)

    dominant_freq = np.argmax(
        fft_values[:len(fft_values)//2]
    )

    dominant_amp = np.max(
        fft_values[:len(fft_values)//2]
    )

    spectral_energy = np.sum(
    fft_values[:N//2]**2
    )
    
    positive_freqs = freqs[:N//2]

    positive_fft = fft_values[:N//2]

    spectral_centroid = (
        np.sum(
            positive_freqs * positive_fft
        )
        /
        np.sum(positive_fft)
    )

    spectral_bandwidth = np.sqrt(
        np.sum(
            ((positive_freqs - spectral_centroid)**2)
            * positive_fft
        )
        /
        np.sum(positive_fft)
    )

    return [
        rms,
        peak,
        crest,
        kurt,
        std,
        mean_abs,
        dominant_freq,
        dominant_amp,
        spectral_energy,
        spectral_centroid,
        spectral_bandwidth
    ]

In [3]:
# Definimos la ruta base donde se encuentran los datos de entrenamiento. En este caso, se asume que los datos están organizados en carpetas por etiqueta (por ejemplo, "normal", "anomalous", etc.)
# y que cada carpeta contiene archivos CSV con las señales de vibración.
# La variable WINDOW_SIZE define el tamaño de la ventana que se utilizará para segmentar las señales de vibración. En este caso, se ha establecido en 1024 muestras, 
# lo que significa que cada segmento de señal que se analizará tendrá 1024 puntos de datos.
# La elección del tamaño de la ventana puede depender de la frecuencia de muestreo de los datos y de la naturaleza de las vibraciones que se están analizando.
# Un tamaño de ventana de 1024 es común en el análisis de vibraciones, ya que proporciona un buen equilibrio entre la resolución temporal y la resolución de frecuencia en el análisis de Fourier.
# La variable dataset se inicializa como una lista vacía, que se llenará con las características extraídas de cada segmento de señal junto con su etiqueta correspondiente.
# El proceso de extracción de características se realiza iterando sobre cada etiqueta en la carpeta base, luego sobre cada archivo CSV dentro de esa etiqueta,
# y finalmente segmentando la señal en ventanas para extraer las características utilizando la función extract_features definida anteriormente.
# Al final de este proceso, la variable dataset contendrá una lista de listas, donde cada sublista representa un conjunto de características extraídas de un segmento de señal junto con su etiqueta.


BASE_DIR = "../data/training"

WINDOW_SIZE = 1024

dataset = []

for label in os.listdir(BASE_DIR):

    folder = os.path.join(
        BASE_DIR,
        label
    )

    if not os.path.isdir(folder):
        continue

    for file in os.listdir(folder):

        if not file.endswith(".csv"):
            continue

        path = os.path.join(
            folder,
            file
        )

        df = pd.read_csv(path)

        df_sensor0 = df[
            df["sensor_id"] == 0
        ]

        df_sensor1 = df[
            df["sensor_id"] == 1
        ]

        signal0_x = df_sensor0["vx_um_s"].values
        signal0_y = df_sensor0["vy_um_s"].values
        signal0_z = df_sensor0["vz_um_s"].values

        signal1_x = df_sensor1["vx_um_s"].values
        signal1_y = df_sensor1["vy_um_s"].values
        signal1_z = df_sensor1["vz_um_s"].values

        for start in range(
            0,
            len(signal0_x)-WINDOW_SIZE,
            WINDOW_SIZE
        ):

            window0_x = signal0_x[start:start+WINDOW_SIZE]
            window0_y = signal0_y[start:start+WINDOW_SIZE]
            window0_z = signal0_z[start:start+WINDOW_SIZE]

            window1_x = signal1_x[start:start+WINDOW_SIZE]
            window1_y = signal1_y[start:start+WINDOW_SIZE]
            window1_z = signal1_z[start:start+WINDOW_SIZE]

            f0x = extract_features(window0_x)
            f0y = extract_features(window0_y)
            f0z = extract_features(window0_z)

            f1x = extract_features(window1_x)
            f1y = extract_features(window1_y)
            f1z = extract_features(window1_z)

            dataset.append(
                f0x +
                f0y +
                f0z +
                f1x +
                f1y +
                f1z +
                [label]
            )

C:\Users\benit\AppData\Local\Temp\ipykernel_26832\1348955913.py:61: RuntimeWarning: invalid value encountered in scalar divide
  np.sum(


In [4]:
# Convertimos la lista de características y etiquetas en un DataFrame de pandas para facilitar su análisis y manipulación.
feature_names = [
    "rms",
    "peak",
    "crest",
    "kurtosis",
    "std",
    "mean_abs",
    "dominant_freq",
    "dominant_amp",
    "spectral_energy",
    "spectral_centroid",
    "spectral_bandwidth"
]

columns = []

for sensor in ["s0", "s1"]:
    for axis in ["x", "y", "z"]:
        columns.extend(
            [
                f"{feat}_{sensor}_{axis}"
                for feat in feature_names
            ]
        )

columns.append("label")

dataset = pd.DataFrame(
    dataset,
    columns=columns
)

dataset.head()

,rms_s0_x,peak_s0_x,crest_s0_x,kurtosis_s0_x,std_s0_x,mean_abs_s0_x,dominant_freq_s0_x,dominant_amp_s0_x,spectral_energy_s0_x,spectral_centroid_s0_x,...,crest_s1_z,kurtosis_s1_z,std_s1_z,mean_abs_s1_z,dominant_freq_s1_z,dominant_amp_s1_z,spectral_energy_s1_z,spectral_centroid_s1_z,spectral_bandwidth_s1_z,label
0,0.000000,0.0,0.000000,NaN,0.000000,0.000000,0,0.000000e+00,0.000000e+00,NaN,...,4.840313,12.802841,9.977859,2.673828,1,2.716670e+03,5.219680e+07,0.081125,0.118088,normal
1,6959.322842,10709.0,1.538799,-1.805022,4438.940983,5359.848633,1,2.754579e+06,1.033061e+13,0.075629,...,2.040499,-1.644046,1706.407900,2002.870117,1,1.085272e+06,1.526636e+12,0.082524,0.123823,normal
2,9065.489082,10989.0,1.212179,-0.537914,1014.413498,9008.554688,4,5.131541e+05,5.395106e+11,0.080227,...,1.658566,0.121616,688.142504,3210.024414,4,2.886054e+05,2.482698e+11,0.085604,0.123676,normal
3,8074.572185,11014.0,1.364035,0.261661,2682.647707,7615.912109,1,1.185932e+06,3.773071e+12,0.077462,...,1.697899,0.388204,569.635957,2711.290039,1,2.214781e+05,1.701210e+11,0.097120,0.129596,normal
4,7413.079071,11325.0,1.527705,-1.522515,3060.033026,6752.032227,1,1.809983e+06,4.909328e+12,0.092848,...,3.900702,34.619584,922.699678,2620.247070,2,2.928789e+05,4.463650e+11,0.109538,0.130450,normal


In [5]:
# Rellenamos los valores faltantes con 0 para evitar problemas en el análisis posterior.
# Esto es importante porque algunos cálculos pueden resultar en valores NaN (por ejemplo, si el RMS es 0, el Crest Factor se vuelve indefinido).
# Al rellenar estos valores con 0, aseguramos que el dataset esté completo y listo para ser utilizado en modelos de machine learning o análisis estadísticos.
dataset = dataset.fillna(0) 

In [6]:
# Guardamos el DataFrame resultante en un archivo CSV para su uso posterior en análisis o modelado.
dataset.to_csv(
    "../dataset_2sensors_xyz.csv",
    index=False
)

print(dataset.shape)

(138, 67)


In [7]:
# Mostramos la distribución de las etiquetas en el dataset para verificar si hay un balance adecuado entre las clases.
dataset["label"].value_counts()

label
unbalance_23g    35
unbalance_27g    35
normal           34
unbalance_19g    34
Name: count, dtype: int64